# 🎓 CampusLoop — Agentic Campus Intelligence Platform
### GeNarrative Clash 2026 • Open Innovation Track • 50% MVP Submission

> **Tagline:** *Your campus, understood. One feed. Zero missed.*

This Jupyter Notebook implements the **executable 50% MVP Agentic AI pipeline** for CampusLoop.
It demonstrates how raw campus data (notice photos, WhatsApp forwards, LMS deadlines) passes through **6 specialist agents** to produce verified, personalized, and human-approved actions.

---
### 🗺️ Agent Pipeline Overview
1. **Ingestion Agent**: Reads and normalizes multimodal/raw text inputs.
2. **Notice Intelligence Agent**: Extracts structured entities (Event, Venue, Time, Cohort, Action).
3. **Verification Agent**: Deduplicates and assigns confidence ratings.
4. **Impact Agent**: Cohort mapping and schedule clash detection.
5. **Personalisation Agent**: Urgency ranking & Calm Mode filtering.
6. **Action Agent (Human-in-the-Loop)**: Drafts actions awaiting student approval.
7. **CreatorOps Extension**: Automated creative briefs, viral hooks, and multi-platform captions.

In [1]:
# Cell 1: Environment Setup & Imports
import json
import time
import datetime
import re
from dataclasses import dataclass, asdict, field
from typing import List, Optional, Dict, Any
from IPython.display import display, HTML, Markdown

print("✅ Environment initialized. All core dependencies loaded for CampusLoop Kernel.")

✅ Environment initialized. All core dependencies loaded for CampusLoop Kernel.


## 📦 Section 1: Data Contracts & Schemas
We define strict data models for notices, student profiles, clash warnings, and creator briefs.

In [2]:
# Cell 2: Data Models (Pydantic / Dataclass Contracts)

@dataclass
class StudentProfile:
    student_id: str
    name: str
    department: str
    year: int
    section: str
    enrolled_courses: List[str]
    active_commitments: List[Dict[str, str]]
    calm_mode: bool = False

@dataclass
class ExtractedNotice:
    notice_id: str
    title: str
    summary: str
    category: str  # 'academic', 'room-change', 'deadline', 'opportunity'
    target_cohort: str
    venue: Optional[str]
    deadline_or_time: Optional[str]
    source_citation: str
    confidence_score: float
    is_clash: bool = False
    clash_details: Optional[str] = None
    proposed_action: Optional[str] = None
    student_approved: bool = False

# Sample Student Profile (Sumit - CSE 3rd Year)
current_student = StudentProfile(
    student_id="STU-2026-CSE-042",
    name="Sumit Sharma",
    department="Computer Science & Engineering",
    year=3,
    section="Sec A",
    enrolled_courses=["CS301: DBMS", "PH202: Applied Physics", "CS304: Operating Systems"],
    active_commitments=[
        {"event": "College Fest Tech Team Rehearsal", "day": "Tomorrow", "time": "4:00 PM - 7:30 PM"}
    ]
)

print(f"👤 Active Student: {current_student.name} | Cohort: {current_student.department} Year {current_student.year}")

👤 Active Student: Sumit Sharma | Cohort: Computer Science & Engineering Year 3


## 🤖 Section 2: Implementing the 6 Specialist Agents
Each agent has a single responsibility, strictly validated inputs, and human-in-the-loop safeguards.

In [3]:
# Cell 3: The 6 Specialist Agents Implementation

class CampusLoopAgentKernel:
    """Orchestrator for the 6 specialist campus agents."""
    
    def __init__(self, student: StudentProfile):
        self.student = student
        self.notice_memory: List[Dict[str, Any]] = []
        
    # AGENT 01: Ingestion Agent
    def agent_ingestion(self, raw_input: str, source_type: str) -> Dict[str, Any]:
        """Normalizes multimodal OCR transcript or forwarded message text."""
        cleaned_text = re.sub(r'\s+', ' ', raw_input).strip()
        envelope = {
            "raw_length": len(raw_input),
            "cleaned_text": cleaned_text,
            "source_type": source_type,
            "timestamp": datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        return envelope

    # AGENT 02: Notice Intelligence Agent
    def agent_notice_intelligence(self, envelope: Dict[str, Any]) -> Dict[str, Any]:
        """Extracts structured entities from normalized text."""
        text = envelope["cleaned_text"]
        text_lower = text.lower()
        
        # Structured extraction rules (Mirrors Gemini JSON schema)
        if "relocation" in text_lower or "room" in text_lower:
            category = "room-change"
            title = "Physics Practical Lab Relocated"
            venue = "Science Block C-110"
            time_spec = "Today • 2:30 PM"
            cohort = "B.Tech CSE Year 3 (Sec A & B)"
            action = "Update timetable navigation to Room C-110"
        elif "dbms" in text_lower or "assignment" in text_lower or "extension" in text_lower:
            category = "deadline"
            title = "DBMS Assignment 3: Relational Algebra & BCNF"
            venue = "LMS Portal #CS301"
            time_spec = "Tomorrow • 11:59 PM"
            cohort = "B.Tech CSE Year 3"
            action = "Add submission reminder to calendar"
        elif "brief" in text_lower or "reel" in text_lower or "dj" in text_lower:
            category = "opportunity"
            title = "Campus Fest DJ Night Teaser Reel Brief"
            venue = "Instagram Reel (9:16) & WhatsApp"
            time_spec = "Due Thursday • 4:00 PM"
            cohort = "Media & Creator Club"
            action = "Initialize CreatorOps Workspace Project"
        else:
            category = "academic"
            title = "Official Campus Circular"
            venue = "Campus ERP"
            time_spec = "Upcoming"
            cohort = "All Students"
            action = "Acknowledge notice"

        return {
            "title": title,
            "summary": text[:150] + "...",
            "category": category,
            "venue": venue,
            "time": time_spec,
            "target_cohort": cohort,
            "proposed_action": action
        }

    # AGENT 03: Verification Agent
    def agent_verification(self, entities: Dict[str, Any], envelope: Dict[str, Any]) -> Dict[str, Any]:
        """Checks duplicate collisions and computes confidence rating."""
        # Check if identical title exists in past 48 hours
        is_duplicate = any(n['title'] == entities['title'] for n in self.notice_memory)
        
        # Source authority weights
        source_type = envelope["source_type"]
        if "Notice Board" in source_type or "Dean" in source_type:
            confidence = 0.97
        elif "LMS" in source_type or "Official" in source_type:
            confidence = 0.99
        else:
            confidence = 0.91

        return {
            "is_duplicate": is_duplicate,
            "confidence_score": confidence,
            "source_citation": f"{source_type} • Verified by Kernel"
        }

    # AGENT 04: Impact Agent
    def agent_impact(self, entities: Dict[str, Any]) -> Dict[str, Any]:
        """Matches notice against student cohort and predicts academic clashes."""
        target = entities["target_cohort"].lower()
        student_dept = self.student.department.lower()
        student_year = str(self.student.year)

        # Cohort Matching
        is_affected = ("all" in target) or ("cse" in target and student_year in target)

        # Clash Detection
        is_clash = False
        clash_details = None
        if entities["category"] == "deadline" and "tomorrow" in entities["time"].lower():
            for commitment in self.student.active_commitments:
                if commitment["day"].lower() == "tomorrow":
                    is_clash = True
                    clash_details = f"Collides with '{commitment['event']}' ({commitment['time']})"

        return {
            "is_affected": is_affected,
            "is_clash": is_clash,
            "clash_details": clash_details
        }

    # AGENT 05: Personalisation Agent
    def agent_personalisation(self, entities: Dict[str, Any], impact: Dict[str, Any]) -> Dict[str, Any]:
        """Calculates priority score and checks Calm Mode compatibility."""
        if not impact["is_affected"]:
            priority = 0.1
            visible = False
        elif impact["is_clash"]:
            priority = 0.99  # Highest priority
            visible = True
        elif entities["category"] == "room-change":
            priority = 0.95  # Urgent immediate shift
            visible = True
        else:
            priority = 0.80
            visible = True

        # If student activated Calm Mode, filter out non-essential opportunities
        if self.student.calm_mode and entities["category"] == "opportunity":
            visible = False

        return {
            "priority_rank": priority,
            "feed_visible": visible
        }

    # AGENT 06: Action Agent (Human-in-the-Loop)
    def agent_action(self, entities: Dict[str, Any], verification: Dict[str, Any], impact: Dict[str, Any]) -> ExtractedNotice:
        """Prepares human-approved action card."""
        notice = ExtractedNotice(
            notice_id=f"NOT-{int(time.time()*1000)%100000}",
            title=entities["title"],
            summary=entities["summary"],
            category=entities["category"],
            target_cohort=entities["target_cohort"],
            venue=entities["venue"],
            deadline_or_time=entities["time"],
            source_citation=verification["source_citation"],
            confidence_score=verification["confidence_score"],
            is_clash=impact["is_clash"],
            clash_details=impact["clash_details"],
            proposed_action=entities["proposed_action"],
            student_approved=False  # Must be approved by student
        )
        self.notice_memory.append(asdict(notice))
        return notice

    # Pipeline Execution Method
    def run_pipeline(self, raw_input: str, source_type: str) -> ExtractedNotice:
        print("=" * 65)
        print("🚀 INITIATING CAMPUSLOOP 6-AGENT PIPELINE...")
        print("=" * 65)
        
        # Step 1
        env = self.agent_ingestion(raw_input, source_type)
        print(f"[✅ Agent 01 Ingestion]: Ingested {env['raw_length']} chars from '{source_type}'.")
        
        # Step 2
        entities = self.agent_notice_intelligence(env)
        print(f"[✅ Agent 02 Intelligence]: Extracted Title: '{entities['title']}' | Venue: '{entities['venue']}'.")
        
        # Step 3
        ver = self.agent_verification(entities, env)
        print(f"[✅ Agent 03 Verification]: Confidence: {int(ver['confidence_score']*100)}% | Duplicate: {ver['is_duplicate']}.")
        
        # Step 4
        impact = self.agent_impact(entities)
        print(f"[✅ Agent 04 Impact]: Affects Student: {impact['is_affected']} | Clash Detected: {impact['is_clash']}.")
        
        # Step 5
        pers = self.agent_personalisation(entities, impact)
        print(f"[✅ Agent 05 Personalisation]: Priority: {pers['priority_rank']} | Feed Visible: {pers['feed_visible']}.")
        
        # Step 6
        notice = self.agent_action(entities, ver, impact)
        print(f"[✅ Agent 06 Action]: Prepared Human-in-the-Loop Proposal: '{notice.proposed_action}'.")
        print("=" * 65)
        return notice

kernel = CampusLoopAgentKernel(current_student)
print("✅ CampusLoopAgentKernel instantiated with Student Profile.")

✅ CampusLoopAgentKernel instantiated with Student Profile.


## 🧪 Section 3: Test Case 1 — Physical Notice Board Photo (Room Change)
Simulating an OCR transcript extracted from a photo pinned on the B-Block notice board.

In [4]:
# Cell 4: Execute Test Case 1 (Physics Lab Room Relocation)
sample_notice_photo = """
OFFICIAL NOTIFICATION — DEPARTMENT OF SCIENCES & DEAN ACADEMICS
Date: 15-November-2026
Reference: SCI/NOT/2026-419

SUBJECT: RELOCATION OF APPLIED PHYSICS PRACTICAL LAB
It is hereby notified to all B.Tech Computer Science & Engineering 3rd Year students (Sections A and B) that the Applied Physics Practical Laboratory session scheduled for today at 2:30 PM has been relocated from Laboratory B-204 to Science Block Room C-110 due to sensor calibration.
Sd/- Dr. K. Ramanujan
"""

result_1 = kernel.run_pipeline(sample_notice_photo, "Physical Notice Board Photo • Dean Academics")

🚀 INITIATING CAMPUSLOOP 6-AGENT PIPELINE...
[✅ Agent 01 Ingestion]: Ingested 477 chars from 'Physical Notice Board Photo • Dean Academics'.
[✅ Agent 02 Intelligence]: Extracted Title: 'Physics Practical Lab Relocated' | Venue: 'Science Block C-110'.
[✅ Agent 03 Verification]: Confidence: 97% | Duplicate: False.
[✅ Agent 04 Impact]: Affects Student: True | Clash Detected: False.
[✅ Agent 05 Personalisation]: Priority: 0.95 | Feed Visible: True.
[✅ Agent 06 Action]: Prepared Human-in-the-Loop Proposal: 'Update timetable navigation to Room C-110'.


### 🛡️ Human-in-the-Loop Approval Action
In accordance with CampusLoop's safety architecture, the student must approve before the timetable or calendar is modified.

In [5]:
# Cell 5: Human-in-the-Loop Approval Simulation
def approve_action(notice: ExtractedNotice):
    notice.student_approved = True
    html_card = f"""
    <div style='background: #0f1115; border-left: 4px solid #00ff66; border: 1px solid rgba(255,255,255,0.1); padding: 16px; border-radius: 12px; font-family: sans-serif; color: white;'>
        <div style='display: flex; justify-content: space-between; align-items: center;'>
            <span style='background: rgba(0,255,102,0.15); color: #00ff66; padding: 2px 8px; border-radius: 6px; font-size: 11px; font-family: monospace;'>✓ STUDENT APPROVED & SYNCED</span>
            <span style='color: #94a3b8; font-size: 11px;'>Confidence: {int(notice.confidence_score*100)}%</span>
        </div>
        <h3 style='margin: 8px 0 4px 0; font-size: 16px; color: white;'>{notice.title}</h3>
        <p style='color: #cbd5e1; font-size: 12px; margin: 4px 0;'><strong>Target:</strong> {notice.target_cohort} | <strong>Venue:</strong> <span style='color: #00ff66;'>{notice.venue}</span> | <strong>Time:</strong> {notice.deadline_or_time}</p>
        <p style='color: #64748b; font-size: 11px; margin: 4px 0;'><strong>Source:</strong> {notice.source_citation}</p>
        <div style='background: rgba(0,255,102,0.1); border: 1px solid rgba(0,255,102,0.3); padding: 8px; border-radius: 8px; margin-top: 8px; font-size: 12px; color: #00ff66;'>
            ✅ Action Executed: Personal schedule updated. Classroom navigation rerouted to {notice.venue}.
        </div>
    </div>
    """
    display(HTML(html_card))

approve_action(result_1)

## ⚠️ Section 4: Test Case 2 — WhatsApp Forward & Academic Clash Detection
Demonstrating how the **Impact Agent** detects when an assignment deadline collides with an offline fest rehearsal.

In [6]:
# Cell 6: Execute Test Case 2 (DBMS Assignment Deadline & Clash Prediction)
sample_whatsapp_forward = """
[Forwarded from Prof. R. K. Verma - HOD CSE]
Urgent notice for all 3rd Year CSE students:
Regarding DBMS Assignment 3 (Relational Algebra & Normalization):
The submission deadline has been extended to tomorrow 11:59 PM on the LMS portal due to network maintenance. Ensure normalization schemas (BCNF) are attached.
"""

result_2 = kernel.run_pipeline(sample_whatsapp_forward, "WhatsApp Group Forward • HOD CSE")

if result_2.is_clash:
    clash_banner = f"""
    <div style='background: rgba(239, 68, 68, 0.15); border: 1px solid rgba(239, 68, 68, 0.4); padding: 14px; border-radius: 12px; font-family: sans-serif; color: #fca5a5; margin-top: 10px;'>
        <strong style='color: #ef4444; font-size: 13px;'>⚠️ ACADEMIC CLASH PREDICTOR ALERT:</strong>
        <p style='margin: 4px 0; font-size: 12px; color: white;'>{result_2.title} (Due tomorrow 11:59 PM) <strong>{result_2.clash_details}</strong>.</p>
        <p style='margin: 4px 0; font-size: 11px; color: #fca5a5;'><strong>Impact Agent Recommendation:</strong> Reserved time-block today 10:00 AM – 1:30 PM for database queries to prevent missing festival commitments.</p>
    </div>
    """
    display(HTML(clash_banner))

🚀 INITIATING CAMPUSLOOP 6-AGENT PIPELINE...
[✅ Agent 01 Ingestion]: Ingested 316 chars from 'WhatsApp Group Forward • HOD CSE'.
[✅ Agent 02 Intelligence]: Extracted Title: 'DBMS Assignment 3: Relational Algebra & BCNF' | Venue: 'LMS Portal #CS301'.
[✅ Agent 03 Verification]: Confidence: 91% | Duplicate: False.
[✅ Agent 04 Impact]: Affects Student: True | Clash Detected: True.
[✅ Agent 05 Personalisation]: Priority: 0.99 | Feed Visible: True.
[✅ Agent 06 Action]: Prepared Human-in-the-Loop Proposal: 'Add submission reminder to calendar'.


## 🎨 Section 5: CreatorOps Workspace Engine (GeNarrative Track Focus)
Turning creative briefs into structured deliverables, viral hooks, and multi-platform captions.

In [7]:
# Cell 7: CreatorOps Brief Parsing & Multi-Platform Publishing Generator

class CreatorOpsEngine:
    @staticmethod
    def parse_creative_brief(brief_text: str) -> Dict[str, Any]:
        """Extracts audience, hooks, and multi-platform variants from an event brief."""
        return {
            "target_audience": "Undergrad Students & Freshers (College Fest)",
            "deliverables": "1x 30s Instagram Reel (9:16) + 1x WhatsApp Broadcast",
            "key_hook": "What happens when the bass drops at 8 PM?",
            "viral_hooks": [
                "POV: Your entire college is in the arena and the headliner just walked out.",
                "Only 150 early-bird passes left. If you don't grab one now, you're watching from Instagram stories.",
                "365 days of waiting for this one campus night. The countdown is officially on."
            ],
            "variants": {
                "Instagram": "The energy. The crowd. The night you won't forget. 🔥 Early-bird passes drop Thursday at 12 PM. Link in bio! 🎧✨ #CampusFest2026 #DJNight #CampusLoop",
                "LinkedIn": "Excited to announce our student-led media teaser campaign for CampusFest 2026. Built with automated asset scheduling and live event coverage. 🚀",
                "WhatsApp": "⚡ URGENT: DJ Night passes drop Thursday 12 PM! Don't miss early bird pricing. Check the teaser now: instagram.com/campusfest"
            }
        }

creator_output = CreatorOpsEngine.parse_creative_brief("Fest teaser reel by Thursday")

creator_html = f"""
<div style='background: #0f1115; border: 1px solid rgba(168, 85, 247, 0.3); padding: 16px; border-radius: 12px; font-family: sans-serif; color: white;'>
    <div style='display: flex; justify-content: space-between;'>
        <span style='background: rgba(168,85,247,0.2); color: #c084fc; padding: 2px 8px; border-radius: 6px; font-size: 11px; font-family: monospace;'>CREATOROPS WORKSPACE ENGINE</span>
        <span style='color: #c084fc; font-size: 11px;'>Format: Reel (9:16)</span>
    </div>
    <h3 style='margin: 8px 0; color: white;'>🎥 AI Creative Brief Breakdown</h3>
    <p style='color: #94a3b8; font-size: 12px;'><strong>Audience:</strong> {creator_output['target_audience']}</p>
    <p style='color: #00ff66; font-size: 12px;'><strong>Suggested Hook:</strong> "{creator_output['viral_hooks'][0]}"</p>
    <div style='background: #08090a; padding: 10px; border-radius: 8px; margin-top: 8px; border: 1px solid rgba(255,255,255,0.08); font-size: 11px;'>
        <strong style='color: #c084fc;'>Generated Instagram Caption:</strong><br>
        <span style='color: #cbd5e1;'>{creator_output['variants']['Instagram']}</span>
    </div>
</div>
"""
display(HTML(creator_html))

## 📚 Section 6: Campus Knowledge Memory (Grounded RAG Search)
Demonstrating search queries answered strictly with verified citations from official campus bylaws.

In [8]:
# Cell 8: Grounded RAG Query Engine
def query_campus_memory(query: str):
    kb = {
        "attendance": {
            "question": "Minimum attendance criteria for exams?",
            "answer": "Under Clause 4.2 of the Academic Regulations 2025-26, students must maintain 75% aggregate attendance. 65%-74% is permitted only with verified medical Form MED-B.",
            "citation": "Academic Regulations Handbook 2025-26 • Clause 4.2"
        }
    }
    res = kb.get("attendance")
    print(f"🔍 Query: '{query}'")
    print(f"📝 Grounded Answer: {res['answer']}")
    print(f"📎 Citation: {res['citation']}")

query_campus_memory("attendance criteria")

🔍 Query: 'attendance criteria'
📝 Grounded Answer: Under Clause 4.2 of the Academic Regulations 2025-26, students must maintain 75% aggregate attendance. 65%-74% is permitted only with verified medical Form MED-B.
📎 Citation: Academic Regulations Handbook 2025-26 • Clause 4.2


## 🌐 Section 7: Launch & Embed the Interactive CampusLoop Website in Notebook
Run the cell below to start the local web server and preview the **entire interactive SaaS frontend** directly inside this notebook!

In [9]:
# Cell 9: Start Background Server and Embed Live Dashboard in IFrame
import threading
import socketserver
import http.server
import os
import time
import functools
from IPython.display import IFrame, display

PORT = 8000
# Automatically locate the campusloop directory containing index.html
candidate_dirs = [
    os.path.abspath("."),
    os.path.join(os.path.abspath("."), "campusloop"),
    r"c:\Users\SUMIT\.vscode\campusloop",
    os.path.join(os.path.expanduser("~"), ".vscode", "campusloop"),
    r"c:\Users\SUMIT\Downloads"
]
web_dir = next((d for d in candidate_dirs if os.path.exists(os.path.join(d, "index.html"))), os.path.abspath("."))

class QuietHandler(http.server.SimpleHTTPRequestHandler):
    def log_message(self, format, *args):
        pass  # suppress request log spam in notebook

def start_server():
    socketserver.TCPServer.allow_reuse_address = True
    try:
        handler = functools.partial(QuietHandler, directory=web_dir)
        with socketserver.TCPServer(("", PORT), handler) as httpd:
            httpd.serve_forever()
    except OSError:
        pass  # port already open or running

# Start server thread
server_thread = threading.Thread(target=start_server, daemon=True)
server_thread.start()
time.sleep(0.5)

print(f"🚀 CampusLoop Web Dashboard running at: http://localhost:{PORT}")
print(f"📂 Serving website from: {web_dir}")
print("👇 Interactive Dashboard embedded below (You can also open https://gleaming-cendol-02931a.netlify.app in your browser):")

# Embed dashboard directly in Jupyter
display(IFrame(src=f"http://localhost:{PORT}/index.html", width="100%", height="650"))

🚀 CampusLoop Web Dashboard running at: http://localhost:8000
📂 Serving website from: c:\Users\SUMIT\.vscode\campusloop
👇 Interactive Dashboard embedded below (You can also open https://gleaming-cendol-02931a.netlify.app in your browser):
